# Qualitative Survey Analysis (Beginner-Friendly Notebook)

This notebook is a step-by-step guided version of the pipeline in `analyze_survey.py`.

Use this when you are new to qualitative NLP and want a clear flow:
1. Install dependencies.
2. Load config.
3. Choose analysis mode.
4. Run analysis.
5. Review and interpret outputs.

## What each mode means

- **General qualitative analysis**: Global clustering + optional sentiment + optional global labels.
- **Thematic analysis by question (unsupervised)**: Themes are discovered separately per survey question.
- **Thematic coding by question (supervised)**: Question-specific label lists are applied using zero-shot classification.
- **Strict coding mode**: Low-confidence assignments are pushed into `UNCERTAIN_REVIEW` for human QA.

In [ ]:
# Run once per environment (or re-run after dependency updates).
%pip install -r ../requirements.txt

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Resolve project root whether notebook is opened from project root or notebooks/.
cwd = Path.cwd().resolve()
if (cwd / "analyze_survey.py").exists():
    ROOT = cwd
elif (cwd.parent / "analyze_survey.py").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not find analyze_survey.py from current notebook location.")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from analyze_survey import load_yaml, run_pipeline

In [ ]:
# Pick your YAML config.
# For first run, keep the example file.
CONFIG_PATH = ROOT / "config" / "config.example.yaml"
config = load_yaml(CONFIG_PATH)

print("Loaded config:", CONFIG_PATH)
config

## Beginner setup checklist

Before running:
1. Confirm `input.file_path` is correct.
2. Confirm `input.text_column` matches your data.
3. If you have multiple prompts/questions, set `input.question_column`.
4. Update zero-shot labels to match your domain.

In [ ]:
# Optional quick edits directly in notebook before running.
# Uncomment lines to override YAML values in-memory for this notebook run.

# Example: change file and output location
# config["input"]["file_path"] = str(ROOT / "data" / "your_survey.csv")
# config["output"]["output_dir"] = str(ROOT / "output")

# Example: global theme labels (for zero-shot)
# config["zero_shot"]["labels"] = [
#     "Usability concern",
#     "Price concern",
#     "Support feedback",
#     "Feature request"
# ]

# Example: thematic analysis by question (unsupervised)
# config["thematic_analysis"]["by_question"]["enabled"] = True

# Example: thematic coding by question (supervised)
# config["thematic_analysis"]["by_question_zero_shot"]["enabled"] = True

# Example: strict coding mode (recommended for production coding)
# config["thematic_analysis"]["by_question_zero_shot"]["strict_coding"]["enabled"] = True
# config["thematic_analysis"]["by_question_zero_shot"]["strict_coding"]["top1_min_score"] = 0.60
# config["thematic_analysis"]["by_question_zero_shot"]["strict_coding"]["min_margin_over_second"] = 0.12

In [ ]:
# Run full pipeline.
# This may take a few minutes on first run because models can be downloaded and cached.
run_pipeline(config)
print("Pipeline run complete.")

In [ ]:
# Load core output files.
output_dir = Path(config["output"]["output_dir"]).expanduser().resolve()
responses = pd.read_csv(output_dir / "responses_scored.csv")
cluster_summary = pd.read_csv(output_dir / "cluster_summary.csv")

print("Output directory:", output_dir)
print("Responses rows:", len(responses))
display(responses.head(10))
display(cluster_summary)

In [ ]:
# Load optional thematic outputs when enabled.
question_theme_path = output_dir / "question_theme_summary.csv"
question_label_path = output_dir / "question_label_summary.csv"
strict_global_path = output_dir / "strict_review_queue_global.csv"
strict_question_path = output_dir / "strict_review_queue_question.csv"

if question_theme_path.exists():
    question_theme = pd.read_csv(question_theme_path)
    print("question_theme_summary.csv loaded")
    display(question_theme.head(20))
else:
    print("question_theme_summary.csv not found (likely disabled).")

if question_label_path.exists():
    question_labels = pd.read_csv(question_label_path)
    print("question_label_summary.csv loaded")
    display(question_labels.head(20))
else:
    print("question_label_summary.csv not found (likely disabled).")

if strict_global_path.exists():
    strict_global = pd.read_csv(strict_global_path)
    print("strict_review_queue_global.csv loaded")
    display(strict_global.head(20))
else:
    print("strict_review_queue_global.csv not found (strict global coding disabled or no uncertain rows).")

if strict_question_path.exists():
    strict_question = pd.read_csv(strict_question_path)
    print("strict_review_queue_question.csv loaded")
    display(strict_question.head(20))
else:
    print("strict_review_queue_question.csv not found (strict question coding disabled or no uncertain rows).")

## How to interpret results (beginner notes)

1. Read `cluster_summary.csv` for high-level themes.
2. Validate themes using representative quotes, not term lists alone.
3. If using question-based analysis, compare themes *within* each question.
4. Review all `UNCERTAIN_REVIEW` rows manually before final reporting.
5. Iterate labels/thresholds and rerun until coding quality is acceptable.

## Output files

- `responses_scored.csv`
- `cluster_summary.csv`
- `silhouette_scores.json`
- `run_report.md`
- `question_theme_summary.csv` (if enabled)
- `question_label_summary.csv` (if enabled)
- `strict_review_queue_global.csv` (if enabled)
- `strict_review_queue_question.csv` (if enabled)